In [1]:
import os, sys
import utils.data_processing as dp
import utils.pair_methods as pm
import scipy
from datetime import datetime
import re
from collections import defaultdict, OrderedDict
#torch
import torch
from torchvision.models import alexnet
from torchvision import transforms as T
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment

#QUBOvert
import qubovert as qv
from qubovert import boolean_var, QUBO,PUBO
from qubovert.sim import anneal_qubo, anneal_pubo
from dwave.samplers import SimulatedAnnealingSampler
from dimod import BinaryQuadraticModel
import dimod
from scipy.linalg import block_diag


In [2]:
#search for and find the files
def path_joiner(image_name, root_dir = None): #recursive search for the image
    if root_dir is None:
        root_dir = os.getcwd()
    for dirpath , _, filenames in os.walk(root_dir):
        if image_name in filenames:
            return os.path.join(dirpath, image_name)

#usage
mat_files = ['Cars_006a.mat','Cars_007a.mat','Cars_008b.mat']
paths = []
for fi in mat_files:
    paths.append(path_joiner(fi))
print(paths)
def keypoint(image_path, max_points=None):
    keypoints1 = scipy.io.loadmat(image_path)["pts_coord"]
    if max_points is not None:
        keypoints1 = keypoints1[:, :max_points]
    return keypoints1

def keypoints_list(image_paths: list, max_points: int):
    keypoints = []
    for image_path in image_paths:
        keypoints1 = keypoint(image_path, max_points)
        keypoints.append(keypoints1)
    return keypoints

def keypoints_dict(image_names: list, max_points: int):
    keypoints = {}
    for image_name in image_names:
        base_name = os.path.splitext(image_name)[0]
        full_path = path_joiner(image_name)
        if full_path:
            kps = keypoint(full_path, max_points)
            keypoints[base_name] = kps
        else:
            print(f"[Warning] image not found: {image_name}")
    return keypoints
points_list = keypoints_list(paths,4)
print(f'This is the keypoints list {points_list}')
points_dic = keypoints_dict(mat_files,max_points=4)
print(f'This is the keypoints dict {points_dic}')

["d:\\University\\Master'sDegree\\Quantum\\10.Quantum Hardware Design and Optimization and Quantum Computing\\Quantum Computing\\project\\QPermutationSynch\\PF-dataset\\car(G)\\Cars_006a.mat", "d:\\University\\Master'sDegree\\Quantum\\10.Quantum Hardware Design and Optimization and Quantum Computing\\Quantum Computing\\project\\QPermutationSynch\\PF-dataset\\car(G)\\Cars_007a.mat", "d:\\University\\Master'sDegree\\Quantum\\10.Quantum Hardware Design and Optimization and Quantum Computing\\Quantum Computing\\project\\QPermutationSynch\\PF-dataset\\car(G)\\Cars_008b.mat"]
This is the keypoints list [array([[ 47.93528064, 151.86426117,  45.965063  ,  69.60767468],
       [ 98.15864834, 121.80126002,  57.2766323 ,  54.81386025]]), array([[ 38.44802   , 128.33906116,  24.28337108,  50.43349215],
       [159.21073186, 177.73373428, 115.62719675, 108.00007811]]), array([[ 24.22058824, 117.125     ,  14.44117647,  24.22058824],
       [167.35845588, 173.22610294, 143.39889706, 128.72977941]])]

In [3]:
#AlexNet feature vectors
def alexnet_feature_extractor(layer= 'conv4'):
    model = alexnet(pretrained=True)
    model.eval()
    if layer == 'conv4':
        return torch.nn.Sequential(*list(model.features)[:10])
    elif layer == 'conv5':
        return torch.nn.Sequential(*list(model.features)[:12])
    else:
        raise ValueError("Invalid layer. Choose 'conv4' or 'conv5'.")
def extract_features(keypoints_dict, patch_size=64, layer='conv4'):
    feature_extractor = alexnet_feature_extractor(layer)
    transform = T.Compose([
        T.Resize((227, 227)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    feature_extractor.to(device)
    features = {}

    for image_name, keypoints in keypoints_dict.items():
        img_path = path_joiner(image_name + '.png')
        img = Image.open(img_path).convert('RGB')
        img_np = np.array(img)
        feature_list = []

        for (x, y) in keypoints.T:
            x, y = int(round(x)), int(round(y))
            x1 = max(0, x - patch_size // 2)
            y1 = max(0, y - patch_size // 2)
            x2 = min(img.width, x + patch_size // 2)
            y2 = min(img.height, y + patch_size // 2)

            patch = img.crop((x1, y1, x2, y2))
            patch_tensor = transform(patch).unsqueeze(0).to(device)
            with torch.no_grad():
                feat = feature_extractor(patch_tensor)
                feat = feat.mean(dim=[2, 3]) # to flatten the output dimensions
            feature_list.append(feat.squeeze().cpu().numpy())

        features[image_name] = np.stack(feature_list)
    return features
features = extract_features(points_dic)
print(features)

c:\Users\sasan\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\sasan\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


{'Cars_006a': array([[0.6327961 , 0.6570724 , 0.43500638, ..., 0.12524807, 0.94820446,
        0.25369614],
       [0.6472052 , 0.7465545 , 0.29996347, ..., 0.31832656, 0.44847858,
        0.7318862 ],
       [0.6550388 , 0.5805156 , 0.519704  , ..., 0.2887706 , 0.6994099 ,
        0.31993425],
       [0.46335208, 0.74790305, 0.71165705, ..., 0.22629021, 0.503425  ,
        0.33397204]], dtype=float32), 'Cars_007a': array([[0.8054076 , 0.31992558, 0.35503426, ..., 0.5955565 , 0.48035055,
        0.25003266],
       [1.2447602 , 0.66127527, 0.3204016 , ..., 0.28860936, 0.679289  ,
        0.43859968],
       [0.7458177 , 0.19756278, 0.6287172 , ..., 0.4006805 , 0.43598127,
        0.30020928],
       [0.7195019 , 0.34268734, 0.98587126, ..., 0.5052143 , 0.19623913,
        0.5717929 ]], dtype=float32), 'Cars_008b': array([[1.331687  , 0.44508156, 0.4737947 , ..., 0.29298073, 0.514822  ,
        0.06485915],
       [1.542289  , 0.71722406, 0.5318643 , ..., 0.26443538, 0.28021538,
       

In [4]:
def pairwise_permutations(features_dict, pm_size: int) -> dict: # compute the P_ij
    P = {}
    image_list = list(features_dict.keys())

    for i in range(len(image_list)):
        key0= f"P{i+1}{i+1}"
        P[key0] = np.eye(pm_size)
        for j in range(i + 1, len(image_list)):
            img1 = image_list[i]
            img2 = image_list[j]

            feats1 = features_dict[img1]
            feats2 = features_dict[img2]
            similarity = cosine_similarity(feats1, feats2)
            cost_matrix = 1 - similarity

            row_ind, col_ind = linear_sum_assignment(cost_matrix)
            n = len(row_ind)
            perm_matrix = np.zeros((n, n), dtype=int)
            perm_matrix[row_ind, col_ind] = 1
            key1 = f"P{i+1}{j+1}"
            P[key1] = perm_matrix
            key2 = f"P{j+1}{i+1}"
            P[key2] = perm_matrix.T
    return P
P = pairwise_permutations(features, 4)
print(len(P))
print(P)

9
{'P11': array([[1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.]]), 'P12': array([[1, 0, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 0, 1]]), 'P21': array([[1, 0, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 0, 1]]), 'P13': array([[0, 0, 0, 1],
       [1, 0, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 1, 0]]), 'P31': array([[0, 1, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 0, 1],
       [1, 0, 0, 0]]), 'P22': array([[1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.]]), 'P23': array([[1, 0, 0, 0],
       [0, 0, 1, 0],
       [0, 1, 0, 0],
       [0, 0, 0, 1]]), 'P32': array([[1, 0, 0, 0],
       [0, 0, 1, 0],
       [0, 1, 0, 0],
       [0, 0, 0, 1]]), 'P33': array([[1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.]])}


In [78]:
#Qubo formulation
# from scipy.linalg import block_diag
# def qubo_formulation(P: dict, num_views: int, num_keypoints: int, penalty: float):
#     # Q matrix size
#     q_size = num_views*(num_keypoints**2)
#     x = [boolean_var(f'x{i}') for i in range(q_size)]
#     x_np_t = np.array(x)
#     x_np = x_np_t.transpose()
#     model = QUBO()
#     #construct Q'
#     Q_blocks = []
#     for i in range(1, num_views + 1):
#         row_blocks = []
#         for j in range(1, num_views + 1):
#             kron = np.kron(np.eye(num_keypoints), P[f'P{i}{j}'])
#             row_blocks.append(kron)
#         Q_blocks.append(row_blocks)

#     Q_prime = -np.block(Q_blocks)  # Shape: (mn^2, mn^2)

#     #construct A matrix of the constraints
#     I = np.eye(num_keypoints)
#     oneT = np.ones((1, num_keypoints))

#     A_list = []
#     b_list = []

#     for _ in range(num_views):
#         A_top = np.kron(I, oneT)       # Row constraints
#         A_bottom = np.kron(oneT, I)    # Column constraints
#         A_i = np.vstack([A_top, A_bottom])  # Shape: (2n, n^2)
#         b_i = np.ones(2 * num_keypoints)
        
#         A_list.append(A_i)
#         b_list.append(b_i)

#     A = block_diag(*A_list)            # Shape: (2mn, mn^2)
#     A_T = A.T
#     b = np.concatenate(b_list)         # Shape: (2mn, )
#     #put together the model
#     Q = Q_prime + penalty*(A_T@A)
#     s = (-2)*penalty*(A_T@b)
#     q_term = x_np_t@Q@x_np
#     model += q_term
#     c_term = (s.T)@x_np
#     model += c_term
#     return model



    

In [5]:
# 2nd QUBO formularion, setting X_1 to identity
from scipy.linalg import block_diag
def qubo_formulation(P: dict, num_views: int, num_keypoints: int, penalty: float):
    n = num_keypoints
    m = num_views
    identity_vec = np.eye(n).flatten()  # vec(I) for fixed X1

    # Define number of free variables
    x_size = (m - 1) * n**2
    x = [boolean_var(f'x{i}') for i in range(x_size)]
    x_np_t = np.array(x)
    x_np = x_np_t.transpose()
    model = QUBO()

    # Build full Q' of shape (mn^2, mn^2)
    Q_blocks = []
    for i in range(1, m + 1):
        row_blocks = []
        for j in range(1, m + 1):
            kron = np.kron(np.eye(n), P[f'P{i}{j}'])
            row_blocks.append(kron)
        Q_blocks.append(row_blocks)

    Q_prime_full = -np.block(Q_blocks)  # (mn^2, mn^2)

    # Split Q' into fixed/free blocks
    Q11 = Q_prime_full[:n**2, :n**2]          # Top-left (X1, X1)
    Q1f = Q_prime_full[:n**2, n**2:]          # Top-right (X1, others)
    Qf1 = Q_prime_full[n**2:, :n**2]          # Bottom-left (others, X1)
    Qff = Q_prime_full[n**2:, n**2:]          # Bottom-right (others, others)

    # Construct constraint matrix A and vector b for free variables only
    A_list = []
    b_list = []
    for _ in range(m - 1):  # only for views 2 to m
        A_top = np.kron(np.eye(n), np.ones((1, n)))       # row constraints
        A_bottom = np.kron(np.ones((1, n)), np.eye(n))    # column constraints
        A_i = np.vstack([A_top, A_bottom])                # (2n, n^2)
        b_i = np.ones(2 * n)
        A_list.append(A_i)
        b_list.append(b_i)

    A = block_diag(*A_list)           # Shape: (2n(m-1), n^2(m-1))
    b = np.concatenate(b_list)        # Shape: (2n(m-1),)

    # Build final QUBO matrices
    Q = Qff + penalty * (A.T @ A)
    s = (-2 * penalty * (A.T @ b)) + (Qf1 + Q1f.T) @ identity_vec

    # Build objective
    q_term = x_np_t @ Q @ x_np
    model += q_term
    c_term = s.T @ x_np
    model += c_term

    return model



In [6]:
num_view = 3
num_keypoints = 4
model = qubo_formulation(P, num_view, num_keypoints,5.0)
print(model)

{('x0',): -13.0, ('x0', 'x1'): 10.0, ('x0', 'x2'): 10.0, ('x0', 'x3'): 10.0, ('x0', 'x4'): 10.0, ('x0', 'x8'): 10.0, ('x0', 'x12'): 10.0, ('x0', 'x16'): -2.0, ('x1',): -11.0, ('x1', 'x2'): 10.0, ('x1', 'x3'): 10.0, ('x1', 'x5'): 10.0, ('x1', 'x9'): 10.0, ('x1', 'x13'): 10.0, ('x1', 'x18'): -2.0, ('x2',): -11.0, ('x2', 'x3'): 10.0, ('x2', 'x6'): 10.0, ('x10', 'x2'): 10.0, ('x14', 'x2'): 10.0, ('x17', 'x2'): -2.0, ('x3',): -11.0, ('x3', 'x7'): 10.0, ('x11', 'x3'): 10.0, ('x15', 'x3'): 10.0, ('x19', 'x3'): -2.0, ('x4',): -11.0, ('x4', 'x5'): 10.0, ('x4', 'x6'): 10.0, ('x4', 'x7'): 10.0, ('x4', 'x8'): 10.0, ('x12', 'x4'): 10.0, ('x20', 'x4'): -2.0, ('x5',): -13.0, ('x5', 'x6'): 10.0, ('x5', 'x7'): 10.0, ('x5', 'x9'): 10.0, ('x13', 'x5'): 10.0, ('x22', 'x5'): -2.0, ('x6',): -11.0, ('x6', 'x7'): 10.0, ('x10', 'x6'): 10.0, ('x14', 'x6'): 10.0, ('x21', 'x6'): -2.0, ('x7',): -11.0, ('x11', 'x7'): 10.0, ('x15', 'x7'): 10.0, ('x23', 'x7'): -2.0, ('x8',): -11.0, ('x8', 'x9'): 10.0, ('x10', 'x8'): 

In [7]:
# use D-wave simulated annealing to solve the model
def dwave_sim_anneal(model, num_trials: int = 1):
    qubo_dwave = model.Q
    sampler = SimulatedAnnealingSampler()
    bqm = BinaryQuadraticModel.from_qubo(qubo_dwave)
    res = sampler.sample(bqm,
        num_reads = num_trials
    )
    best_sample = res.first.sample
    best_energy = res.first.energy
    return best_sample,best_energy, res
best_sample, best_energy, sim_res = dwave_sim_anneal(model,100)
print(best_sample)
print(best_energy)

{'x0': np.int8(1), 'x1': np.int8(0), 'x10': np.int8(1), 'x11': np.int8(0), 'x12': np.int8(0), 'x13': np.int8(0), 'x14': np.int8(0), 'x15': np.int8(1), 'x16': np.int8(1), 'x17': np.int8(0), 'x18': np.int8(0), 'x19': np.int8(0), 'x2': np.int8(0), 'x20': np.int8(0), 'x21': np.int8(0), 'x22': np.int8(1), 'x23': np.int8(0), 'x24': np.int8(0), 'x25': np.int8(1), 'x26': np.int8(0), 'x27': np.int8(0), 'x28': np.int8(0), 'x29': np.int8(0), 'x3': np.int8(0), 'x30': np.int8(0), 'x31': np.int8(1), 'x4': np.int8(0), 'x5': np.int8(1), 'x6': np.int8(0), 'x7': np.int8(0), 'x8': np.int8(0), 'x9': np.int8(0)}
-106.0


In [8]:
def extract_permutation_matrices(best_sample: dict, num_views: int, num_keypoints: int):
    matrices = {}
    n = num_keypoints
    n2 = n ** 2

    matrices['X1'] = np.eye(n, dtype=int)

    for v in range(1, num_views):  # start from 1 (i.e., X2, X3, ...)
        X = np.zeros((n, n), dtype=int)
        base_idx = (v - 1) * n2  # shift because X1 is skipped
        for i in range(n):
            for j in range(n):
                var_idx = base_idx + i * n + j
                var_name = f'x{var_idx}'
                X[i, j] = best_sample.get(var_name, 0)
        matrices[f'X{v+1}'] = X

    return matrices
#
matrices = extract_permutation_matrices(best_sample, num_view, num_keypoints)
print(matrices)

{'X1': array([[1, 0, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 0, 1]]), 'X2': array([[1, 0, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 0, 1]]), 'X3': array([[1, 0, 0, 0],
       [0, 0, 1, 0],
       [0, 1, 0, 0],
       [0, 0, 0, 1]])}


In [9]:
def is_valid_permutation_matrix(X): #check the validity of the absolute permutation matrices(all rows and columns sum to 1)
    return (
        np.all(np.sum(X, axis=0) == 1) and
        np.all(np.sum(X, axis=1) == 1) and
        np.all((X == 0) | (X == 1))
    )

for k, i in matrices.items():
    print(f'matrix {k} is valid : {is_valid_permutation_matrix(i)}')

matrix X1 is valid : True
matrix X2 is valid : True
matrix X3 is valid : True


In [118]:
import inspect
print(inspect.getsource(dp.get_all_keypoints))

def get_all_keypoints(*file_paths):
    all_coords =[]
    for path in file_paths:
        coords1, _ = pair_coordinates(path,path)
        all_coords.append(coords1)
    return all_coords



In [119]:
import importlib
import utils.data_processing as dp
importlib.reload(dp)

<module 'utils.data_processing' from "d:\\University\\Master'sDegree\\Quantum\\10.Quantum Hardware Design and Optimization and Quantum Computing\\Quantum Computing\\project\\QPermutationSynch\\utils\\data_processing.py">

In [15]:
data_path1 = r'./PF-dataset/car(G)/Cars_006a.png'
data_path2 = r'./PF-dataset/car(G)/Cars_007a.png'
data_path3 = r'./PF-dataset/car(G)/Cars_008b.png'
image_paths = [data_path1, data_path2, data_path3]



In [16]:
import os
from datetime import datetime
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import matplotlib.lines as mlines
import random
import string

# Helper to generate visually distinct colors
def generate_n_colors(n):
    cmap = plt.cm.get_cmap("tab10", n)
    return [cmap(i) for i in range(n)]

# Main visualization and result-saving function
def visualize_and_save_result(points_list, permutation_matrices, mat_files, best_energy, penalty_value, output_dir="./results"):
    os.makedirs(output_dir, exist_ok=True)

    num_views = len(points_list)
    num_keypoints = points_list[0].shape[1]

    # Construct matched keypoints indices across views using the permutation matrices
    # X1 = identity, we use its keypoint indices as the base
    matches = [[] for _ in range(num_keypoints)]  # each inner list will contain 1 index per image

    for i in range(num_keypoints):
        matches[i].append(i)  # X1 is identity

    for view_idx in range(1, num_views):
        X = permutation_matrices[f'X{view_idx+1}']
        for i in range(num_keypoints):
            target = np.argmax(X[i])
            matches[i].append(target)

    # Load images
    images = []
    for mat_file in mat_files:
        img_path = path_joiner(mat_file.replace('.mat', '.png'))  # assumes .png format
        images.append(Image.open(img_path).convert("RGB"))

    # Create canvas for side-by-side image
    widths, heights = zip(*(img.size for img in images))
    total_width = sum(widths)
    max_height = max(heights)

    canvas = Image.new('RGB', (total_width, max_height), color=(255, 255, 255))
    x_offsets = []
    x_offset = 0
    for img in images:
        canvas.paste(img, (x_offset, 0))
        x_offsets.append(x_offset)
        x_offset += img.size[0]

    # Plot keypoints
    fig, ax = plt.subplots(figsize=(16, 4))
    ax.imshow(canvas)
    ax.axis('off')

    colors = generate_n_colors(num_keypoints)
    radius = 5

    for keypoint_idx, match in enumerate(matches):
        color = colors[keypoint_idx]
        prev_x, prev_y = None, None
        for view_idx, kp_idx in enumerate(match):
            pt = points_list[view_idx][:, kp_idx]
            x = x_offsets[view_idx] + pt[0]
            y = pt[1]
            circle = plt.Circle((x, y), radius, color=color, fill=True)
            ax.add_patch(circle)
            if prev_x is not None:
                line = mlines.Line2D([prev_x, x], [prev_y, y], color=color, linewidth=2)
                ax.add_line(line)
            prev_x, prev_y = x, y

    # Save image and text file
    short_names = "_".join([os.path.splitext(f)[0].lower() for f in mat_files])
    timestamp = datetime.now().strftime("%H%M%S")
    base_filename = f"{short_names}_{timestamp}"
    image_save_path = os.path.join(output_dir, base_filename + ".png")
    txt_save_path = os.path.join(output_dir, base_filename + ".txt")

    fig.savefig(image_save_path, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)

    with open(txt_save_path, "w") as f:
        f.write(f"Images: {mat_files}\n")
        f.write(f"Penalty: {penalty_value}\n")
        f.write(f"Energy: {best_energy}\n")
        f.write(f"Permutation Matrices:\n")
        for k, mat in permutation_matrices.items():
            f.write(f"{k}:\n{mat}\n\n")

    return image_save_path, txt_save_path

In [18]:
#
image_path, txt_path = visualize_and_save_result(
    points_list=points_list,
    permutation_matrices=matrices,
    mat_files=mat_files,
    best_energy=best_energy,
    penalty_value=5.0  # or your actual value
)
print(f"Saved image to: {image_path}")
print(f"Saved text to: {txt_path}")

Saved image to: ./results\cars_006a_cars_007a_cars_008b_152513.png
Saved text to: ./results\cars_006a_cars_007a_cars_008b_152513.txt


C:\Users\sasan\AppData\Local\Temp\ipykernel_43176\2230953731.py:12: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap("tab10", n)
